###**Programación Concurrente**
####Actividad Práctica (Opcional) - Comunicación y Sincronización

---

##**Ejercicio 1 - Carga Criolla S.A.**

##Para ejecutar código en C# en Google Colab

1.   **Instalar .NET**: Instalar el SDK de .NET en el entorno de Google Colab para poder compilar y ejecutar programas en C#.
2.   **Crear el proyecto C#**: Utilizar dotnet new console para crear un proyecto de consola y escribir el código en el archivo Program.cs.
3.   **Compilar el programa**: Utilizar dotnet build para compilar el proyecto y generar los archivos necesarios para su ejecución.
4.   **Ejecutar el programa**: Utilizar dotnet run para ejecutar el programa directamente desde una celda de Google Colab.

In [ ]:
!wget -q https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb > /dev/null 2>&1
!dpkg -i packages-microsoft-prod.deb > /dev/null 2>&1
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y dotnet-sdk-8.0 -qq > /dev/null 2>&1
!echo "✅ .NET instalado correctamente"

✅ .NET instalado correctamente


In [ ]:
!dotnet new console -n AP1Parte3 --force

=========
Welcome to .NET 8.0!
---------------------
SDK Version: 8.0.425

Telemetry
---------
The .NET tools collect usage data in order to help us improve your experience. It is collected by Microsoft and shared with the community. You can opt-out of telemetry by setting the DOTNET_CLI_TELEMETRY_OPTOUT environment variable to '1' or 'true' using your favorite shell.

Read more about .NET CLI Tools telemetry: https://aka.ms/dotnet-cli-telemetry

----------------
Installed an ASP.NET Core HTTPS development certificate.
To trust the certificate, view the instructions: https://aka.ms/dotnet-https-linux

----------------
Write your first app: https://aka.ms/dotnet-hello-world
Find out what's new: https://aka.ms/dotnet-whats-new
Explore documentation: https://aka.ms/dotnet-docs
Report issues and find source on GitHub: https://github.com/dotnet/core
Use 'dotnet --help' to see available commands or visit: https://aka.ms/dotnet-cli
----------------------------------------------------

In [ ]:
%%writefile AP1Parte3/CargaCriolla.cs
using System;
using System.Collections.Generic;
using System.Threading;
using System.Threading.Tasks;

class Truck
{
    public int Id { get; }
    public string Location { get; set; }

    public Truck(int id)
    {
        Id = id;
        Location = "Buenos Aires";
    }
}

class Simulation
{
    // Parámetros de simulación
    private const double DEFAULT_SIMULATION_HOUR_IN_MILLISECONDS = 10;
    private const int HOURS_PER_DAY = 24;

    // Constantes de duración (en horas simuladas)
    private const double LOADING_UNLOADING_DURATION = 2;
    private const double REFUELING_DURATION = 1;
    private const double NO_TRIP_WAIT_DURATION = 0.5;
    private const int MIN_TRAVEL_HOURS = 18;
    private const int MAX_TRAVEL_HOURS = 25;

    // Capacidades de los semáforos
    private const int SINGLE_CAPACITY = 1;
    private const int FERNANDEZ_FUEL_STATIONS_CAPACITY = 2;

    private readonly List<Truck> trucks = new();

    // Número de viajes pendientes de cada planta
    private int tripsFromBuenosAires;
    private int tripsFromFernandez;

    // Protege el acceso a los contadores de viajes compartidos
    private readonly object tripLock = new();

    // Mantiene el orden de la salida de los mensajes
    private readonly object consoleLock = new();

    // Un camión puede cargar en cada planta a la vez
    private readonly SemaphoreSlim buenosAiresLoading = new(SINGLE_CAPACITY, SINGLE_CAPACITY);
    private readonly SemaphoreSlim fernandezLoading = new(SINGLE_CAPACITY, SINGLE_CAPACITY);

    // Un camión puede descargar en cada planta a la vez
    private readonly SemaphoreSlim buenosAiresUnloading = new(SINGLE_CAPACITY, SINGLE_CAPACITY);
    private readonly SemaphoreSlim fernandezUnloading = new(SINGLE_CAPACITY, SINGLE_CAPACITY);

    // Fernández tiene dos surtidores de combustible
    private readonly SemaphoreSlim fuelStations = new(FERNANDEZ_FUEL_STATIONS_CAPACITY, FERNANDEZ_FUEL_STATIONS_CAPACITY);

    private readonly Random random = new();

    private readonly double simulationHourInMilliseconds;

    private DateTime startTime;

    public Simulation(
        int numberOfTrucks,
        int numberOfTrips,
        double simulationHourInMilliseconds = DEFAULT_SIMULATION_HOUR_IN_MILLISECONDS)
    {
        tripsFromBuenosAires = numberOfTrips;
        tripsFromFernandez = numberOfTrips;

        this.simulationHourInMilliseconds =
            simulationHourInMilliseconds;

        for (int i = 1; i <= numberOfTrucks; i++)
        {
            trucks.Add(new Truck(i));
        }
    }

    public void Run()
    {
        startTime = DateTime.Now;

        PrintMessage(
            "============================================"
        );

        PrintMessage(
            "Carga Criolla S.A. - Truck Simulation"
        );

        PrintMessage(
            "============================================"
        );

        PrintMessage(
            $"Trucks: {trucks.Count}"
        );

        PrintMessage(
            $"Trips from Buenos Aires: {tripsFromBuenosAires}"
        );

        PrintMessage(
            $"Trips from Fernandez: {tripsFromFernandez}"
        );

        PrintMessage("");

        List<Task> truckTasks = new();

        foreach (Truck truck in trucks)
        {
            Truck currentTruck = truck;

            truckTasks.Add(
                Task.Run(() => TruckRoutine(currentTruck))
            );
        }

        Task.WaitAll(truckTasks.ToArray());

        TimeSpan elapsed = DateTime.Now - startTime;

        PrintMessage("");
        PrintMessage("============================================");
        PrintMessage("Simulation finished");
        PrintMessage("============================================");
        PrintMessage(
            $"Real execution time: {elapsed.TotalSeconds:F2} seconds"
        );

        double simulatedDays =
            elapsed.TotalMilliseconds /
            (simulationHourInMilliseconds * HOURS_PER_DAY);

        PrintMessage(
            $"Simulated time: {simulatedDays:F2} days"
        );

        PrintMessage("============================================");
    }

    private void TruckRoutine(Truck truck)
    {
        while (true)
        {
            string? departure;

            lock (tripLock)
            {
                // Verifica si hay viajes disponibles
                // desde la ubicación actual del camión.
                if (truck.Location == "Buenos Aires" &&
                    tripsFromBuenosAires > 0)
                {
                    tripsFromBuenosAires--;
                    departure = "Buenos Aires";
                }
                else if (truck.Location == "Fernandez" &&
                         tripsFromFernandez > 0)
                {
                    tripsFromFernandez--;
                    departure = "Fernandez";
                }
                else if (tripsFromBuenosAires == 0 &&
                         tripsFromFernandez == 0)
                {
                    // No quedan viajes en ningún lugar.
                    return;
                }
                else
                {
                    departure = null;
                }
            }

            // No hay viajes disponibles actualmente en la ubicación de este camión.
            if (departure == null)
            {
                SimulatedWait(NO_TRIP_WAIT_DURATION);
                continue;
            }

            if (departure == "Buenos Aires")
            {
                ExecuteBuenosAiresToFernandez(truck);
            }
            else
            {
                ExecuteFernandezToBuenosAires(truck);
            }
        }
    }

    private void ExecuteBuenosAiresToFernandez(Truck truck)
    {
        PrintTruckState(
            truck,
            "Waiting for loading in Buenos Aires"
        );

        buenosAiresLoading.Wait();

        try
        {
            PrintTruckState(
                truck,
                "Loading flour in Buenos Aires"
            );

            SimulatedWait(LOADING_UNLOADING_DURATION);

            PrintTruckState(
                truck,
                "Finished loading flour"
            );
        }
        finally
        {
            buenosAiresLoading.Release();
        }

        PrintTruckState(
            truck,
            "Departing loaded towards Fernandez"
        );

        int travelHours = GetTravelTime();

        SimulatedWait(travelHours);

        truck.Location = "Fernandez";

        PrintTruckState(
            truck,
            $"Arrived at Fernandez after {travelHours} hours"
        );

        PrintTruckState(
            truck,
            "Waiting for unloading in Fernandez"
        );

        fernandezUnloading.Wait();

        try
        {
            PrintTruckState(
                truck,
                "Unloading flour in Fernandez"
            );

            SimulatedWait(LOADING_UNLOADING_DURATION);

            PrintTruckState(
                truck,
                "Finished unloading flour"
            );
        }
        finally
        {
            fernandezUnloading.Release();
        }

        PrintTruckState(
            truck,
            "Ready to perform a Fernandez -> Buenos Aires trip"
        );
    }

    private void ExecuteFernandezToBuenosAires(Truck truck)
    {
        PrintTruckState(
            truck,
            "Waiting for loading in Fernandez"
        );

        fernandezLoading.Wait();

        try
        {
            PrintTruckState(
                truck,
                "Loading charcoal in Fernandez"
            );

            SimulatedWait(LOADING_UNLOADING_DURATION);

            PrintTruckState(
                truck,
                "Finished loading charcoal"
            );
        }
        finally
        {
            fernandezLoading.Release();
        }

        PrintTruckState(
            truck,
            "Waiting for a diesel fuel station"
        );

        fuelStations.Wait();

        try
        {
            PrintTruckState(
                truck,
                "Refueling with diesel in Fernandez"
            );

            SimulatedWait(REFUELING_DURATION);

            PrintTruckState(
                truck,
                "Finished refueling"
            );
        }
        finally
        {
            fuelStations.Release();
        }

        PrintTruckState(
            truck,
            "Departing loaded towards Buenos Aires"
        );

        int travelHours = GetTravelTime();

        SimulatedWait(travelHours);

        truck.Location = "Buenos Aires";

        PrintTruckState(
            truck,
            $"Arrived at Buenos Aires after {travelHours} hours"
        );

        PrintTruckState(
            truck,
            "Waiting for unloading in Buenos Aires"
        );

        buenosAiresUnloading.Wait();

        try
        {
            PrintTruckState(
                truck,
                "Unloading charcoal in Buenos Aires"
            );

            SimulatedWait(LOADING_UNLOADING_DURATION);

            PrintTruckState(
                truck,
                "Finished unloading charcoal"
            );
        }
        finally
        {
            buenosAiresUnloading.Release();
        }

        PrintTruckState(
            truck,
            "Ready for another Buenos Aires -> Fernandez trip"
        );
    }

    private int GetTravelTime()
    {
        lock (random)
        {
            return random.Next(MIN_TRAVEL_HOURS, MAX_TRAVEL_HOURS);
        }
    }

    private void SimulatedWait(double hours)
    {
        int milliseconds =
            (int)(hours * simulationHourInMilliseconds);

        Thread.Sleep(milliseconds);
    }

    private void PrintTruckState(Truck truck, string state)
    {
        lock (consoleLock)
        {
            TimeSpan elapsed =
                DateTime.Now - startTime;

            double simulatedHours =
                elapsed.TotalMilliseconds /
                simulationHourInMilliseconds;

            double simulatedDays =
                simulatedHours / HOURS_PER_DAY;

            Console.WriteLine(
                $"[{simulatedDays,6:F2} days] " +
                $"Truck {truck.Id,2} | " +
                $"{state}"
            );
        }
    }

    private void PrintMessage(string message)
    {
        lock (consoleLock)
        {
            Console.WriteLine(message);
        }
    }
}

class Program
{
    static void Main(string[] args)
    {
        if (args.Length != 2)
        {
            Console.WriteLine(
                "Usage: dotnet run -- <number_of_trucks> <number_of_trips>"
            );

            return;
        }

        if (!int.TryParse(args[0], out int numberOfTrucks) ||
            !int.TryParse(args[1], out int numberOfTrips))
        {
            Console.WriteLine(
                "Both parameters must be integers."
            );

            return;
        }

        if (numberOfTrucks <= 0 ||
            numberOfTrips <= 0)
        {
            Console.WriteLine(
                "Both parameters must be greater than zero."
            );

            return;
        }

        Simulation simulation =
            new Simulation(
                numberOfTrucks,
                numberOfTrips
            );

        simulation.Run();
    }
}


Overwriting AP1Parte3/CargaCriolla.cs


In [ ]:
!dotnet run --project AP1Parte3 -- 5 10

==============================================
Carga Criolla S.A. - Truck Simulation
Trucks: 5
Trips from Buenos Aires: 10
Trips from Fernandez: 10

[  0.17 days] Truck  1 | Waiting for loading in Buenos Aires
[  0.20 days] Truck  2 | Waiting for loading in Buenos Aires
[  0.20 days] Truck  2 | Loading flour in Buenos Aires
[  0.29 days] Truck  2 | Finished loading flour
[  0.29 days] Truck  2 | Departing loaded towards Fernandez
[  0.29 days] Truck  1 | Loading flour in Buenos Aires
[  0.38 days] Truck  1 | Finished loading flour
[  0.38 days] Truck  1 | Departing loaded towards Fernandez
[  1.13 days] Truck  1 | Arrived at Fernandez after 18 hours
[  1.13 days] Truck  1 | Waiting for unloading in Fernandez
[  1.13 days] Truck  1 | Unloading flour in Fernandez
[  1.17 days] Truck  2 | Arrived at Fernandez after 21 hours
[  1.17 days] Truck  2 | Waiting for unloading in Fernandez
[  1.22 days] Truck  1 | Finished unloading flour
[  1.22 days] Truck  1 | Ready to perform a Fernandez -

In [ ]:
!dotnet run --project AP1Parte3 -- 10 50

==============================================
Carga Criolla S.A. - Truck Simulation
Trucks: 10
Trips from Buenos Aires: 50
Trips from Fernandez: 50

[  0.11 days] Truck  1 | Waiting for loading in Buenos Aires
[  0.13 days] Truck  1 | Loading flour in Buenos Aires
[  0.13 days] Truck  2 | Waiting for loading in Buenos Aires
[  0.22 days] Truck  1 | Finished loading flour
[  0.22 days] Truck  1 | Departing loaded towards Fernandez
[  0.22 days] Truck  2 | Loading flour in Buenos Aires
[  0.30 days] Truck  2 | Finished loading flour
[  0.30 days] Truck  2 | Departing loaded towards Fernandez
[  1.09 days] Truck  2 | Arrived at Fernandez after 19 hours
[  1.09 days] Truck  2 | Waiting for unloading in Fernandez
[  1.09 days] Truck  2 | Unloading flour in Fernandez
[  1.18 days] Truck  1 | Arrived at Fernandez after 23 hours
[  1.18 days] Truck  1 | Waiting for unloading in Fernandez
[  1.18 days] Truck  2 | Finished unloading flour
[  1.18 days] Truck  2 | Ready to perform a Fernandez 

In [ ]:
!dotnet run --project AP1Parte3 -- 1 10

==============================================
Carga Criolla S.A. - Truck Simulation
Trucks: 1
Trips from Buenos Aires: 10
Trips from Fernandez: 10

[  0.10 days] Truck  1 | Waiting for loading in Buenos Aires
[  0.12 days] Truck  1 | Loading flour in Buenos Aires
[  0.20 days] Truck  1 | Finished loading flour
[  0.20 days] Truck  1 | Departing loaded towards Fernandez
[  1.20 days] Truck  1 | Arrived at Fernandez after 24 hours
[  1.20 days] Truck  1 | Waiting for unloading in Fernandez
[  1.20 days] Truck  1 | Unloading flour in Fernandez
[  1.29 days] Truck  1 | Finished unloading flour
[  1.29 days] Truck  1 | Ready to perform a Fernandez -> Buenos Aires trip
[  1.29 days] Truck  1 | Waiting for loading in Fernandez
[  1.29 days] Truck  1 | Loading charcoal in Fernandez
[  1.37 days] Truck  1 | Finished loading charcoal
[  1.38 days] Truck  1 | Waiting for a diesel fuel station
[  1.38 days] Truck  1 | Refueling with diesel in Fernandez
[  1.42 days] Truck  1 | Finished refuelin

In [ ]:
!dotnet run --project AP1Parte3 -- 0 10

==Both parameters must be greater than zero.
=

In [ ]:
!dotnet run --project AP1Parte3 -- 10 10

==============================================
Carga Criolla S.A. - Truck Simulation
Trucks: 10
Trips from Buenos Aires: 10
Trips from Fernandez: 10

[  0.09 days] Truck  1 | Waiting for loading in Buenos Aires
[  0.11 days] Truck  2 | Waiting for loading in Buenos Aires
[  0.11 days] Truck  2 | Loading flour in Buenos Aires
[  0.19 days] Truck  2 | Finished loading flour
[  0.19 days] Truck  2 | Departing loaded towards Fernandez
[  0.20 days] Truck  1 | Loading flour in Buenos Aires
[  0.28 days] Truck  1 | Finished loading flour
[  0.28 days] Truck  1 | Departing loaded towards Fernandez
[  1.03 days] Truck  2 | Arrived at Fernandez after 20 hours
[  1.03 days] Truck  2 | Waiting for unloading in Fernandez
[  1.03 days] Truck  2 | Unloading flour in Fernandez
[  1.11 days] Truck  2 | Finished unloading flour
[  1.12 days] Truck  2 | Ready to perform a Fernandez -> Buenos Aires trip
[  1.12 days] Truck  2 | Waiting for loading in Fernandez
[  1.12 days] Truck  2 | Loading charcoal 

In [ ]:
!dotnet run --project AP1Parte3 -- 5 1000

Se han truncado las últimas 5000 líneas del flujo de salida.
[330.47 days] Truck  5 | Unloading charcoal in Buenos Aires
[330.56 days] Truck  5 | Finished unloading charcoal
[330.56 days] Truck  5 | Ready for another Buenos Aires -> Fernandez trip
[330.56 days] Truck  5 | Waiting for loading in Buenos Aires
[330.56 days] Truck  3 | Finished loading flour
[330.56 days] Truck  4 | Unloading charcoal in Buenos Aires
[330.56 days] Truck  3 | Departing loaded towards Fernandez
[330.56 days] Truck  5 | Loading flour in Buenos Aires
[330.64 days] Truck  4 | Finished unloading charcoal
[330.64 days] Truck  1 | Arrived at Buenos Aires after 21 hours
[330.64 days] Truck  1 | Waiting for unloading in Buenos Aires
[330.64 days] Truck  1 | Unloading charcoal in Buenos Aires
[330.64 days] Truck  5 | Finished loading flour
[330.64 days] Truck  5 | Departing loaded towards Fernandez
[330.64 days] Truck  4 | Ready for another Buenos Aires -> Fernandez trip
[330.64 days] Truck  4 | Waiting for loading i

In [ ]:
!dotnet run --project AP1Parte3 -- 500 100

==============================================
Carga Criolla S.A. - Truck Simulation
Trucks: 500
Trips from Buenos Aires: 100
Trips from Fernandez: 100

[  0.09 days] Truck  1 | Waiting for loading in Buenos Aires
[  0.11 days] Truck  1 | Loading flour in Buenos Aires
[  0.11 days] Truck  2 | Waiting for loading in Buenos Aires
[  0.20 days] Truck  1 | Finished loading flour
[  0.20 days] Truck  1 | Departing loaded towards Fernandez
[  0.20 days] Truck  2 | Loading flour in Buenos Aires
[  0.28 days] Truck  2 | Finished loading flour
[  0.28 days] Truck  2 | Departing loaded towards Fernandez
[  1.20 days] Truck  1 | Arrived at Fernandez after 24 hours
[  1.20 days] Truck  1 | Waiting for unloading in Fernandez
[  1.20 days] Truck  1 | Unloading flour in Fernandez
[  1.28 days] Truck  2 | Arrived at Fernandez after 24 hours
[  1.28 days] Truck  2 | Waiting for unloading in Fernandez
[  1.29 days] Truck  1 | Finished unloading flour
[  1.29 days] Truck  1 | Ready to perform a Fernand

### Conclusiones
A partir de las pruebas del ejercicio, analizamos cómo la concurrencia impacta directamente en la sincronización y en la performance global del sistema. Para proteger los recursos compartido, como los contadores de viajes, hubo que definir regiones críticas y aplicar exclusión mutua mediante locks, mientras que las plantas y los surtidores se modelaron con semáforos contadores para limitar el acceso simultáneo.
En términos de rendimiento, observamos que aumentar la cantidad de camiones reduce el tiempo total solo hasta cierto punto, ya que las capacidades fijas de carga, descarga y combustible actúan como cuellos de botella. Cuando la concurrencia supera la capacidad de estos recursos compartidos, los hilos pasan la mayor parte del tiempo bloqueados en colas de espera, generando sobrecarga innecesaria en la planificación del SO.
Concluimos que maximizar la performance no consiste en añadir entidades concurrentes de forma indiscriminada, sino en balancear el número de hilos con los puntos de sincronización.   